# Import Data

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random
import warnings
import torch
import multiprocessing
import os


class CFG:
    
    target_name='tag'
    seed = 58
    
    #######################################################################################
    # GPU
    gpu_available = torch.cuda.is_available()

    print(f"CUDA available: {gpu_available}")
    if gpu_available:
        print(f"GPU name: {torch.cuda.get_device_name(0)}")
        print(f"Number of GPUs: {torch.cuda.device_count()}")
    else:
        print(f'Use CPU, \nNumber of CPUs: {multiprocessing.cpu_count()}')
    
    #######################################################################################
    # Seed
    
    @staticmethod
    def seed_all(seed=42):
        random.seed(seed)
        np.random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


CFG.seed_all(CFG.seed)

warnings.simplefilter('ignore')
sns.set_theme(style="ticks")

CUDA available: True
GPU name: Tesla T4
Number of GPUs: 2


In [3]:
x = pd.read_csv('/kaggle/input/datasets/artsmirnovch/spd-feature-set-9/x_train.csv')
y = pd.read_csv('/kaggle/input/datasets/artsmirnovch/spd-feature-set-9/y_train.csv')
x = x.drop('Unnamed: 0', axis=1)
y = y.drop('Unnamed: 0', axis=1)

# Prepare data

In [4]:
from sklearn.model_selection import train_test_split


x_train, x_val, y_train, y_val = train_test_split(
    x, y, stratify=y, shuffle=True, test_size=0.2, random_state=CFG.seed
)

# Optuna

In [5]:
import optuna

from xgboost import XGBClassifier

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def objective(trial):

    if CFG.gpu_available:
        device = "cuda"
        n_jobs_setting = 1
    else:
        device = "cpu"
        n_jobs_setting = -1

    param_space = {
        'tree_method': 'hist',
        'booster': 'gbtree',
        'grow_policy': 'depthwise',
        'eval_metric': 'auc',
        'random_state': CFG.seed,
        'use_label_encoder': False,
        'verbosity': 0,
        'device': device,
        'n_jobs': n_jobs_setting,
        'max_bin': 256,
        'early_stopping_rounds': trial.suggest_int('early_stopping_rounds', 10, 200, step=10),
        'n_estimators': trial.suggest_int('n_estimators', 100, 3000, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'eta': trial.suggest_float('eta', 1e-4, 1e-1, log=True),
        'lambda': trial.suggest_float('lambda', 1e-5, 2, log=True),
        'alpha': trial.suggest_float('alpha', 1e-5, 2, log=True),
        'gamma': trial.suggest_float('gamma', 0, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
        'max_delta_step': trial.suggest_float('max_delta_step', 0, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1),
        'colsample_bynode': trial.suggest_float('colsample_bynode', 0.5, 1),
    }

    # Var 1
    # local_x_train, local_x_val, local_y_train, local_y_val = train_test_split(
    #     x, y, stratify=y, shuffle=True, test_size=0.25
    # )
    # clf = XGBClassifier(**params).fit(local_x_train, local_y_train, eval_set=[(local_x_val, local_y_val)], verbose=0)
    # y_pred_proba = clf.predict_proba(local_x_val)[:, 1]
    # auc_score = roc_auc_score(local_y_val, y_pred_proba)
    # return {'score': auc_score, 'status': STATUS_OK}
    
    # Var 2
    seed = param_space['random_state'] # solve ModuleNotFoundError
    
    model = XGBClassifier(**param_space)
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    split_generator = skf.split(x_train, y_train)
    
    train_scores = []
    val_scores = []
    for fold, (train_idx, val_idx) in enumerate(split_generator):
        
        fold_x_train = x_train.iloc[train_idx, :]
        fold_y_train = y_train.iloc[train_idx]
        fold_x_val = x_train.iloc[val_idx, :]
        fold_y_val = y_train.iloc[val_idx]
        
        fold_model = XGBClassifier(**param_space)
        
        fold_model.fit(
            fold_x_train, 
            fold_y_train,
            eval_set=[(fold_x_train, fold_y_train), (fold_x_val, fold_y_val)],
            verbose=0
        )
        
        fold_train_pred_proba = fold_model.predict_proba(fold_x_train)[:, 1]
        fold_val_pred_proba = fold_model.predict_proba(fold_x_val)[:, 1]
        
        fold_train_score = roc_auc_score(fold_y_train, fold_train_pred_proba)
        fold_val_score = roc_auc_score(fold_y_val, fold_val_pred_proba)
        
        train_scores.append(fold_train_score)
        val_scores.append(fold_val_score)

    train_scores = np.array(train_scores)
    val_scores = np.array(val_scores)

    trial.set_user_attr("train_scores_mean", train_scores.mean())

    # return val_scores.mean() - 4 * abs(val_scores.mean() - train_scores.mean())
    return val_scores.mean()


study = optuna.create_study(
    direction='maximize', 
    study_name='xgb_optimization',
    load_if_exists=True
)

study.optimize(objective, n_trials=600, timeout=3.5*3600, n_jobs=-1, show_progress_bar=True)

[I 2026-02-27 04:05:00,130] A new study created in memory with name: xgb_optimization


  0%|          | 0/600 [00:00<?, ?it/s]

[I 2026-02-27 04:05:08,774] Trial 3 finished with value: 0.9229093368989242 and parameters: {'early_stopping_rounds': 10, 'n_estimators': 1150, 'max_depth': 5, 'eta': 0.00026342894855656534, 'lambda': 0.0038599064657308694, 'alpha': 0.003987363768427084, 'gamma': 0.9842453331170262, 'min_child_weight': 2.5170872333088945, 'max_delta_step': 2.773821879258851, 'subsample': 0.6687223395999936, 'colsample_bytree': 0.5075672366834183, 'colsample_bylevel': 0.9348784314446278, 'colsample_bynode': 0.9429156168711249}. Best is trial 3 with value: 0.9229093368989242.
[I 2026-02-27 04:06:30,605] Trial 2 finished with value: 0.9427415465401378 and parameters: {'early_stopping_rounds': 40, 'n_estimators': 1350, 'max_depth': 9, 'eta': 0.0004335334412932911, 'lambda': 0.00016845066078562828, 'alpha': 0.004440680926486607, 'gamma': 0.8025067864399208, 'min_child_weight': 2.9684687604534226, 'max_delta_step': 6.311775399105639, 'subsample': 0.5995389244734384, 'colsample_bytree': 0.8081493933214584, 'c

In [8]:
best_trial = study.best_trial
print(f"Best validation score: {best_trial.value}")
print(f"Best train score: {best_trial.user_attrs['train_scores_mean']}")

best_params = study.best_params
print("Best parameters:", best_params)

Best validation score: 0.9554094821104183
Best train score: 0.9872463674024885
Best parameters: {'early_stopping_rounds': 170, 'n_estimators': 2450, 'max_depth': 9, 'eta': 0.006598468387339539, 'lambda': 0.7158862176191002, 'alpha': 0.013728069097027173, 'gamma': 0.20423851636886323, 'min_child_weight': 6.051131996437884, 'max_delta_step': 1.8668429628525331, 'subsample': 0.7193597182161883, 'colsample_bytree': 0.8894992402619911, 'colsample_bylevel': 0.713979551713394, 'colsample_bynode': 0.7208299645199266}


In [9]:
import plotly.graph_objects as go


trials_df = study.trials_dataframe()
val_scores = trials_df['value'].values
train_scores = [t.user_attrs['train_scores_mean'] for t in study.trials]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(len(val_scores))),
    y=val_scores,
    mode='markers+lines',
    name='Validation Score',
    marker={'color': 'blue'}
))

fig.add_trace(go.Scatter(
    x=list(range(len(train_scores))),
    y=train_scores,
    mode='markers+lines',
    name='Train Score',
    marker={'color': 'red'}
))

fig.update_layout(
    title='Optimization History - Train vs Validation Scores',
    xaxis_title='Trial',
    yaxis_title='AUC Score',
    hovermode='x unified'
)

fig.show()

In [10]:
fig.write_html("optimization_history.html")